# 04 - Avaliação independente e anotação
Avaliação integral, modos de intervenção, monitor independente e exportação cega. Requer execuções próprias já treinadas. Sem dados, não produz tabelas fictícias.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess
cwd = Path.cwd()
candidates = [cwd, cwd.parent, cwd / "revisao_2026", cwd.parent / "revisao_2026",
              cwd / "cot-obfuscation-h2a" / "revisao_2026"]
ROOT = next((p for p in candidates if (p / "src/instruments.py").is_file()), None)
if ROOT is None:
    clone = cwd / "cot-obfuscation-h2a"
    if not clone.exists():
        subprocess.run(["git", "clone", "https://github.com/Kelvin-Marques-Cyber/cot-obfuscation-h2a.git",
                        str(clone)], check=True)
    ROOT = clone / "revisao_2026"
assert (ROOT / "src/instruments.py").is_file(), ROOT
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
ENV = dict(os.environ, PYTHONPATH=str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""))
from notebook_runtime import run_logged

def run_step(command, log_name):
    # Atualiza credenciais/variáveis configuradas depois da célula inicial.
    current_env = dict(os.environ)
    current_env["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
    current_env["PYTHONUNBUFFERED"] = "1"
    return run_logged(command, ROOT / "logs" / log_name, cwd=ROOT, env=current_env)

print("Raiz:", ROOT)


In [ ]:
RUN_ROOT = (ROOT / "runs/grpo").resolve()
DATA_PATH = ROOT / "data/grpo_synthetic.jsonl"
statuses = sorted(RUN_ROOT.glob("*/status.json"))
trained = [p.parent for p in statuses if json.loads(p.read_text()).get("status") == "trained"]
print("Execuções treinadas encontradas:", len(trained))
EXECUTAR_AVALIACAO = False
if EXECUTAR_AVALIACAO:
    assert trained, "Nenhum adaptador treinado."
    for run_dir in trained:
        for base_only in [True, False]:
            args = [sys.executable, "-m", "gpu_lab.evaluate", "--run-dir", str(run_dir),
                    "--data", str(DATA_PATH), "--budget", "384"]
            if base_only:
                args.append("--base-only")
            run_step(args, f"grpo_eval_{run_dir.name}_base{base_only}.log")


In [ ]:
from gpu_lab.analysis import analyze, export_annotations, independent_probe
HAS_GENERATIONS = any(RUN_ROOT.rglob("evaluation_*.jsonl"))
if HAS_GENERATIONS:
    display(analyze(RUN_ROOT, ROOT / "results/new_models"))
    print("Itens para anotação:", export_annotations(RUN_ROOT, ROOT / "annotation"))
    print("Sonda textual:", independent_probe(RUN_ROOT, ROOT / "results/new_models/probe.json"))
else:
    print("Ainda não há gerações; execute as células de avaliação após o treino.")


## Monitor independente
Use identidade diferente da usada na recompensa. O avaliador não deve receber nomes das condições. Verifique falsos positivos e sensibilidade contra o comportamento observável, sem interpretar automaticamente como intenção.


In [ ]:
from gpu_lab.independent_monitor import evaluate_monitor
EXECUTAR_MONITOR = False
INDEPENDENT_MODEL = os.environ.get("INDEPENDENT_MONITOR_MODEL", "")
INDEPENDENT_REVISION = os.environ.get("INDEPENDENT_MONITOR_REVISION", "")
INDEPENDENT_BASE_URL = os.environ.get("INDEPENDENT_MONITOR_BASE_URL", "")
if EXECUTAR_MONITOR:
    assert all([INDEPENDENT_MODEL, INDEPENDENT_REVISION, INDEPENDENT_BASE_URL])
    for source in sorted(RUN_ROOT.rglob("evaluation_*.jsonl")):
        cfg = json.loads((source.parent / "manifest.json").read_text())["resolved_config"]
        assert INDEPENDENT_MODEL != cfg.get("monitor_model"), "Use monitor independente."
        evaluate_monitor(source, source.with_name("independent_" + source.name),
            INDEPENDENT_MODEL, INDEPENDENT_REVISION, INDEPENDENT_BASE_URL)


## Concordância humana
Preencha cópias de blinded_annotations.csv com rater e label. Não distribua private_annotation_key.csv aos avaliadores. O alfa pode ser negativo e pode ser indefinido com ausência de variação.


In [ ]:
from gpu_lab.analysis import nominal_alpha
import csv
ANNOTATIONS = ROOT / "annotation/completed_annotations.csv"
if ANNOTATIONS.exists():
    with ANNOTATIONS.open() as f:
        annotations = list(csv.DictReader(f))
    print("Alfa nominal:", nominal_alpha(annotations))
else:
    print("Nenhuma anotação humana foi fornecida.")


A agregação descritiva e a comparação early/full não substituem o modelo hierárquico, o estudo de poder e as análises de equivalência do protocolo. O artigo não é atualizado automaticamente para evitar incorporar coleta exploratória como confirmatória.
